In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/bench/checkpoints/pre_cell_35.pickle

In [ ]:
%%cudf.pandas.profile
### cell 35 ###

# --- Fix: define the question variable and rename 2018 columns ---
question_of_interest_old_cell47 = (
    'What machine learning frameworks have you used in the past 5 years?'
)
question_of_interest_cell47 = (
    'Which of the following machine learning frameworks do you use on a regular basis?'
)
# align 2018 columns to the new question text
responses_df_2018_cell10.columns = (
    responses_df_2018_cell10.columns
        .str.replace(
            question_of_interest_old_cell47,
            question_of_interest_cell47,
            regex=False
        )
)

# --- optimized for cudf.pandas extension ---

def grab_subset_of_data_47(df, question_of_interest):
    subset = df.filter(like=question_of_interest)
    # split on the last '-' and strip only leading whitespace to keep any intentional trailing spaces
    rename_map = {col: col.rsplit('-', 1)[-1].lstrip() for col in subset.columns}
    return subset.rename(columns=rename_map).dropna(how='all')


def combine_subsets(question, include_2017=False):
    year_dfs = [
        ('2018', responses_df_2018_cell10),
        ('2019', responses_df_2019_cell10),
        ('2020', responses_df_2020),
        ('2021', responses_df_2021),
        ('2022', responses_df_2022_cell10),
    ]
    if include_2017:
        year_dfs.insert(0, ('2017', responses_df_2017))
    subsets = [grab_subset_of_data_47(df, question) for _, df in year_dfs]
    years = [yr for yr, _ in year_dfs]
    # concat with year as a key, then reset into a column
    df_combined = pd.concat(subsets, keys=years, names=['year'])
    df_combined = df_combined.reset_index(level='year').reset_index(drop=True)
    df_counts = df_combined.groupby('year', as_index=False).count()
    return df_combined, df_counts


def convert_df_of_counts_to_percentages_47(df, df_counts):
    # compute denominators of each year directly on the combined df
    denom = df['year'].value_counts().sort_index()
    pct = (
        df_counts.astype(int)
                 .set_index('year')
                 .div(denom, axis=0)
                 * 100
    )
    return pct.reset_index()

# --- main pipeline for cell 35 ---
(ml_df_combined_cell47, ml_df_combined_counts_cell47) = combine_subsets(
    question_of_interest_cell47,
    include_2017=False
)

# bulk combine framework columns
ml_df_combined_2_cell47 = ml_df_combined_cell47.copy()
combos = [
    (['TensorFlow', 'TensorFlow ', 'Keras', 'Keras '],
     'TensorFlow/Keras', 'TensorFlow/Keras'),
    (['PyTorch', 'PyTorch ', 'PyTorch Lightning ', 'Fast.ai ', 'Fastai'],
     'PyTorch/Lightning/Fast.ai', 'PyTorch/PyTorch Lightning/Fast.ai'),
    (['Xgboost', 'Xgboost ', 'lightgbm', 'LightGBM ', 'catboost', 'CatBoost '],
     'Xgboost/LightGBM/CatBoost', 'Xgboost/LightGBM/CatBoost'),
    (['Scikit—learn ', 'Scikit—learn ', 'learn ', 'Learn'],
     'Scikit-learn', 'Scikit-learn')
]
for cols, new_col, label in combos:
    # safely select any of the combo columns that exist
    mask = ml_df_combined_2_cell47.filter(items=cols).notna().any(axis=1)
    # map boolean mask to the combined label or None
    ml_df_combined_2_cell47[new_col] = mask.map({True: label, False: None})
    # drop original columns, ignoring any that weren't present
    ml_df_combined_2_cell47 = ml_df_combined_2_cell47.drop(columns=cols, errors='ignore')

# recalc counts & percentages
ml_df_combined_counts_2_cell47 = (
    ml_df_combined_2_cell47.groupby('year', as_index=False)
                           .count()
)
ml_df_combined_percentages_cell47 = convert_df_of_counts_to_percentages_47(
    ml_df_combined_2_cell47,
    ml_df_combined_counts_2_cell47
)

# select & order output columns
ml_df_combined_percentages_cell47 = ml_df_combined_percentages_cell47[
    ['year', 'Scikit-learn', 'TensorFlow/Keras',
     'PyTorch/Lightning/Fast.ai', 'Xgboost/LightGBM/CatBoost']
]

# melt & sort
df_cell47 = ml_df_combined_percentages_cell47.melt(
    id_vars=['year'],
    value_vars=[
        'Scikit-learn', 'Xgboost/LightGBM/CatBoost',
        'TensorFlow/Keras', 'PyTorch/Lightning/Fast.ai'
    ],
    var_name='variable',
    value_name='value'
).sort_values(['year', 'value']).reset_index(drop=True)
# rename the variable column to blank
df_cell47 = df_cell47.rename(columns={'variable': ''})
# inspect result
df_cell47.info()